
Ground Truth Statistics Scanner

Scans all ground truth JSON files and collects statistics on:
- isolates_with_linking
- isolate_without_linking
- no_isolates_only_assayinformation

Author: Luqman
Project: AI6129 Pathogen Tracking


In [10]:
import json
import os
from pathlib import Path
from typing import Any
from google.colab import drive

In [11]:
def is_empty(data: Any) -> bool:
    """
    Check if data is empty (handles dict, list, None).

    Args:
        data: The data to check

    Returns:
        True if data is empty or None, False otherwise
    """
    if data is None:
        return True
    if isinstance(data, dict) and len(data) == 0:
        return True
    if isinstance(data, list) and len(data) == 0:
        return True
    return False


def count_entries(section_data: Any) -> int:
    """
    Counts entries depending on data type.

    Args:
        section_data: The section data (dict or list)

    Returns:
        Number of entries in the section
    """
    if section_data is None:
        return 0
    if isinstance(section_data, dict):
        return len(section_data.keys())
    if isinstance(section_data, list):
        return len(section_data)
    return 0

def extract_pmcid(file_path: Path, data: dict) -> str:
    """
    Extract PMCID from data or filename.

    Args:
        file_path: Path to the JSON file
        data: Loaded JSON data

    Returns:
        PMCID string
    """
    # Try to get from data first
    if "pmcid" in data:
        return data["pmcid"]

    # Fall back to filename
    return file_path.stem


def initialise_stats() -> dict:
    """
    Initialise the statistics dictionary structure.

    Returns:
        Empty statistics dictionary
    """
    sections = [
        "isolates_with_linking",
        "isolate_without_linking",
        "no_isolates_only_assayinformation"
    ]

    stats = {
        "total_files": 0,
        "files_with_errors": 0,
        "error_files": []
    }

    for section in sections:
        stats[section] = {
            "files_with_data": 0,
            "files_empty": 0,
            "files_missing_key": 0,
            "total_entries": 0,
            "pmcids_with_data": []
        }

    return stats

In [12]:
def scan_ground_truth_statistics(ground_truth_directory: str) -> dict:
    """
    Scans all JSON files in ground truth directory and collects statistics
    on the three target sections.

    Args:
        ground_truth_directory: Path to the ground truth folder

    Returns:
        Dictionary containing statistics for each section
    """
    # Initialise variables
    stats = initialise_stats()
    ground_truth_path = Path(ground_truth_directory)
    sections_to_check = [
        "isolates_with_linking",
        "isolate_without_linking",
        "no_isolates_only_assayinformation"
    ]

    # Get list of all JSON files
    json_files = list(ground_truth_path.glob("*.json"))
    json_files.sort()

    print(f"Found {len(json_files)} JSON files in {ground_truth_directory}")
    print("-" * 60)

    # Process each file
    for file_path in json_files:
        stats["total_files"] += 1

        # Load JSON content
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Error reading {file_path.name}: {e}")
            stats["files_with_errors"] += 1
            stats["error_files"].append(file_path.name)
            continue
        except Exception as e:
            print(f"Unexpected error reading {file_path.name}: {e}")
            stats["files_with_errors"] += 1
            stats["error_files"].append(file_path.name)
            continue

        pmcid = extract_pmcid(file_path, data)

        # Check each section
        for section in sections_to_check:
            if section not in data:
                stats[section]["files_missing_key"] += 1
            else:
                section_data = data[section]
                if is_empty(section_data):
                    stats[section]["files_empty"] += 1
                else:
                    stats[section]["files_with_data"] += 1
                    stats[section]["total_entries"] += count_entries(section_data)
                    stats[section]["pmcids_with_data"].append(pmcid)

    return stats

def print_statistics(stats: dict) -> None:
  """
  Formats and displays the statistics summary.

  Args:
      stats: The statistics dictionary
  """
  sections = [
      "isolates_with_linking",
      "isolate_without_linking",
      "no_isolates_only_assayinformation"
  ]

  print("")
  print("=" * 70)
  print("GROUND TRUTH STATISTICS SUMMARY")
  print("=" * 70)
  print(f"Total files scanned: {stats['total_files']}")
  print(f"Files with errors: {stats['files_with_errors']}")

  if stats["error_files"]:
      print(f"Error files: {', '.join(stats['error_files'])}")

  print("")
  print("-" * 70)

  # Detailed breakdown for each section
  for section in sections:
      section_stats = stats[section]
      print(f"\nSection: {section}")
      print(f"  Files with data:       {section_stats['files_with_data']}")
      print(f"  Files empty:           {section_stats['files_empty']}")
      print(f"  Files missing key:     {section_stats['files_missing_key']}")
      print(f"  Total entries:         {section_stats['total_entries']}")

  # Summary table
  print("")
  print("-" * 70)
  print("SUMMARY TABLE")
  print("-" * 70)
  print(f"{'Section':<40} | {'With Data':>10} | {'Empty':>6} | {'Missing':>8} | {'Entries':>8}")
  print("-" * 70)

  for section in sections:
      section_stats = stats[section]
      print(f"{section:<40} | {section_stats['files_with_data']:>10} | "
            f"{section_stats['files_empty']:>6} | {section_stats['files_missing_key']:>8} | "
            f"{section_stats['total_entries']:>8}")

  print("-" * 70)

In [13]:
def print_pmcids_with_data(stats: dict, section: str) -> None:
    """
    Print the list of PMCIDs that have data for a given section.

    Args:
        stats: The statistics dictionary
        section: The section name to display PMCIDs for
    """
    if section not in stats:
        print(f"Section '{section}' not found in statistics.")
        return

    pmcids = stats[section]["pmcids_with_data"]
    print(f"\nPMCIDs with data in '{section}' ({len(pmcids)} files):")
    print("-" * 40)

    for pmcid in pmcids:
        print(f"  {pmcid}")


def export_to_csv(stats: dict, output_path: str) -> None:
    """
    Export statistics to a CSV file.

    Args:
        stats: The statistics dictionary
        output_path: Path for the output CSV file
    """
    import csv

    sections = [
        "isolates_with_linking",
        "isolate_without_linking",
        "no_isolates_only_assayinformation"
    ]

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Write header
        writer.writerow([
            "Section",
            "Files_With_Data",
            "Files_Empty",
            "Files_Missing_Key",
            "Total_Entries"
        ])

        # Write data rows
        for section in sections:
            section_stats = stats[section]
            writer.writerow([
                section,
                section_stats["files_with_data"],
                section_stats["files_empty"],
                section_stats["files_missing_key"],
                section_stats["total_entries"]
            ])

        # Write summary row
        writer.writerow([])
        writer.writerow(["Total Files Scanned", stats["total_files"]])
        writer.writerow(["Files With Errors", stats["files_with_errors"]])

    print(f"\nStatistics exported to: {output_path}")

In [14]:
def main():
    """
    Main execution function.
    """


    # Configuration
    ground_truth_path = "/content/drive/MyDrive/AI6129/ground_truth"
    output_csv_path = "/content/drive/MyDrive/AI6129/assay_ground_truth_statistics.csv"

    # Run scanner
    stats = scan_ground_truth_statistics(ground_truth_path)

    # Print results
    print_statistics(stats)

    # Optionally print PMCIDs for each section
    # Uncomment the lines below if needed
    print_pmcids_with_data(stats, "isolates_with_linking")
    print_pmcids_with_data(stats, "isolate_without_linking")
    print_pmcids_with_data(stats, "no_isolates_only_assayinformation")

    # Export to CSV
    export_to_csv(stats, output_csv_path)

    return stats


if __name__ == "__main__":
    # Mount google drive
    drive.mount('/content/drive', force_remount=False)
    stats = main()

Mounted at /content/drive
Found 227 JSON files in /content/drive/MyDrive/AI6129/ground_truth
------------------------------------------------------------

GROUND TRUTH STATISTICS SUMMARY
Total files scanned: 227
Files with errors: 0

----------------------------------------------------------------------

Section: isolates_with_linking
  Files with data:       122
  Files empty:           105
  Files missing key:     0
  Total entries:         724

Section: isolate_without_linking
  Files with data:       114
  Files empty:           113
  Files missing key:     0
  Total entries:         10807

Section: no_isolates_only_assayinformation
  Files with data:       21
  Files empty:           206
  Files missing key:     0
  Total entries:         254

----------------------------------------------------------------------
SUMMARY TABLE
----------------------------------------------------------------------
Section                                  |  With Data |  Empty |  Missing |  Entries
